In [ ]:
pip install pandas numpy statsmodels scikit-learn openpyxl scipy

In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.2 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3

import os
import math
import warnings
from typing import Tuple, Dict

import numpy as np
import pandas as pd

# statsmodels for GLMs and count models
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.discrete_model import NegativeBinomial
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# ------------------------- CONFIG (edit these paths) -------------------------
import os

# 1. Sesuaikan folder. Di Colab biasanya defaultnya "/content" (tanpa titik di depan)
DATA_DIR = "/content"

FILES = {
    "cargo": {
        "file": os.path.join(DATA_DIR, "cargo_loss.xlsx"),
        "freq_sheet": "Frequency",
        "sev_sheet": "Severity",
    },
    "equipment": {
        "file": os.path.join(DATA_DIR, "equipment_failure.xlsx"),
        "freq_sheet": "Frequency",
        "sev_sheet": "Severity",
    },
    "worker": {
        "file": os.path.join(DATA_DIR, "worker_compensation.xlsx"),
        "freq_sheet": "Frequency",
        "sev_sheet": "Severity",
    },
    "bi": {
        "file": os.path.join(DATA_DIR, "business_interruption.xlsx"),
        "freq_sheet": "Frequency",
        "sev_sheet": "Severity",
    }
}
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Actuarial loadings (tune these)
LOADINGS = {
    "expense_ratio": 0.20,  # operational expense
    "commission": 0.05,  # distribution commission
    "profit_loading": 0.05,  # target profit/risk margin
    "reinsurance_loading": 0.05,  # ceded costs & reinsurance premium
}

# Inflation history provided by user (years 2160-2174). We'll read into a df in code below.
INFLATION_HISTORY = {
    2160: 0.0377,
    2161: 0.0232,
    2162: 0.0148,
    2163: 0.0108,
    2164: 0.0022,
    2165: 0.0071,
    2166: 0.0155,
    2167: 0.0216,
    2168: 0.0181,
    2169: 0.0073,
    2170: 0.0294,
    2171: 0.0708,
    2172: 0.0598,
    2173: 0.0276,
    2174: 0.0239,
}

# Company financials (Đ million) 2172-2174 (from user) -- used for exposure growth estimate
COMPANY_FINANCIALS = pd.DataFrame(
    {
        "year": [2172, 2173, 2174],
        "net_revenue": [55259, 60303, 61491],
        "operating_income": [14574, 17893, 19263],
        "net_income": [11116, 12441, 14755],
    }
)

# Projection years
PROJECTION_YEARS = [2175, 2176, 2177, 2178, 2179]

# Risk classification quantiles
RISK_QUANTILES = [0.33, 0.66]

# Safety caps
MAX_CLAIM_LIMIT_MULTIPLIER = 5.0  # limit per claim won't exceed 5x mean claim amount by default

# ------------------------- Utility functions -------------------------

def read_excel_safe(path: str, sheet: str) -> pd.DataFrame:

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_excel(path, sheet_name=sheet)

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    return df

def ensure_numeric(df: pd.DataFrame, cols: list):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")


# ------------------------- Modeling helpers -------------------------

def fit_frequency_negative_binomial(df: pd.DataFrame, formula: str, maxiter=100) -> Tuple[object, str]:
    """Try Negative Binomial; fallback to Poisson with overdispersion correction."""
    try:
        model = NegativeBinomial.from_formula(formula, df)
        res = model.fit(disp=False, maxiter=maxiter)
        return res, "nb"
    except Exception as e:
        # fallback
        try:
            glm_poisson = smf.glm(formula=formula, data=df, family=sm.families.Poisson()).fit()
            # estimate dispersion by Pearson chi2 / df
            pearson_chi2 = glm_poisson.pearson_chi2
            df_resid = glm_poisson.df_resid
            dispersion = pearson_chi2 / df_resid if df_resid > 0 else 1.0
            glm_poisson.dispersion = dispersion
            return glm_poisson, "poisson_disp"
        except Exception as e2:
            raise RuntimeError("Both NB and Poisson frequency models failed: " + str(e2))


def fit_severity_gamma(df: pd.DataFrame, formula: str) -> Tuple[object, str]:
    df_pos = df[df['claim_amount'] > 0].copy()
    if df_pos.shape[0] < 20:
        # too few observations -> fallback to empirical mean
        return None, "empirical_mean"
    try:
        glm_gamma = smf.glm(formula=formula, data=df_pos, family=sm.families.Gamma(sm.families.links.log())).fit()
        return glm_gamma, "gamma"
    except Exception:
        # fallback to lognormal (OLS on log)
        try:
            df_pos['log_claim'] = np.log(df_pos['claim_amount'])
            ols = smf.ols(formula=formula.replace('claim_amount', 'log_claim'), data=df_pos).fit()
            return ols, "lognormal_ols"
        except Exception:
            return None, "empirical_mean"


# ------------------------- Pricing logic per product -------------------------

def classify_risk(pure_prem_series: pd.Series) -> pd.Series:
    q_low = pure_prem_series.quantile(RISK_QUANTILES[0])
    q_high = pure_prem_series.quantile(RISK_QUANTILES[1])
    def cls(v):
        if v <= q_low:
            return 'low'
        elif v <= q_high:
            return 'medium'
        else:
            return 'high'
    return pure_prem_series.apply(cls)


def compute_gross_premium(pure_premium: float, loadings: Dict[str, float]) -> float:
    total_loading = sum(loadings.values())
    if total_loading >= 1.0:
        raise ValueError("Total loadings too large")
    return pure_premium / (1 - total_loading)


def suggest_limits_from_severity(df_sev: pd.DataFrame, severity_predictor: object, segment_key: str, value_key: str) -> pd.DataFrame:
    """Return suggested per-claim limit and annual aggregate by segment. Conservative approach: per-claim = min(99th percentile, MAX_MULT * mean).
    Annual limit = max(3x median annualised claims, 5x per-claim) depending on business logic. """
    out = []
    grouped = df_sev.groupby(segment_key)
    for seg, g in grouped:
        mean_claim = g['claim_amount'].mean()
        p99 = g['claim_amount'].quantile(0.99)
        per_claim = min(p99 if not np.isnan(p99) else mean_claim * 3, mean_claim * MAX_CLAIM_LIMIT_MULTIPLIER)
        # annual limit: use 3x mean annual claim * exposure estimate (we'll set per-nasabah so exposure=1)
        annual_limit = max(per_claim * 3, mean_claim * 10)
        out.append({segment_key: seg, 'mean_claim': mean_claim, 'p99': p99, 'suggested_per_claim_limit': per_claim, 'suggested_annual_limit': annual_limit})
    return pd.DataFrame(out)

def clean_formula_terms(terms):
    clean = []
    for t in terms:
        if t.startswith("C(") and t.endswith(")"):
            clean.append(t[2:-1])  # ambil isi dalam C()
        else:
            clean.append(t)
    return clean
# ------------------------- Projection helpers -------------------------

def estimate_inflation_projection(years: list, inflation_history: Dict[int, float]) -> Dict[int, float]:
    # use mean inflation of recent 5 years (2170-2174) as default; but compute if available
    recent = [inflation_history.get(y) for y in range(max(inflation_history.keys()) - 4, max(inflation_history.keys()) + 1)]
    recent = [x for x in recent if x is not None]
    if len(recent) == 0:
        base = 0.025
    else:
        base = float(np.mean(recent))
    proj = {}
    for y in years:
        # we simply apply base for each future year; users can replace with custom series
        proj[y] = base
    return proj


def estimate_exposure_growth(company_financials: pd.DataFrame) -> float:
    # compute revenue CAGR from earliest to latest and return annual growth
    df = company_financials.sort_values('year')
    start = df.iloc[0]['net_revenue']
    end = df.iloc[-1]['net_revenue']
    n = df.iloc[-1]['year'] - df.iloc[0]['year']
    if start <= 0 or n <= 0:
        return 0.0
    cagr = (end / start) ** (1.0 / n) - 1.0
    return float(cagr)


def project_loss_ratios(pp_per_policy: pd.Series, n_policies_by_segment: pd.Series, freq_pred_by_segment: pd.Series, sev_pred_by_segment: pd.Series, proj_years: list, inflation_proj: Dict[int, float], exposure_growth: float, loadings: Dict[str, float]) -> pd.DataFrame:
    # For each projection year compute expected claims and earned premium
    rows = []
    # baseline exposure (number policies) per segment
    base_policies = n_policies_by_segment
    for y in proj_years:
        # escalate severity by inflation
        years_from_now = y - min(proj_years)
        # cumulative inflation factor (approx exp sum)
        cum_infl = (1 + inflation_proj[y]) ** years_from_now if years_from_now > 0 else 1.0
        # escalate exposure by compounding exposure_growth
        exposure_factor = (1 + exposure_growth) ** (y - proj_years[0]) if y - proj_years[0] > 0 else 1.0
        expected_claims = 0.0
        expected_premium = 0.0
        for seg in pp_per_policy.index:
            policies = base_policies.loc[seg]
            freq = freq_pred_by_segment.loc[seg]
            sev = sev_pred_by_segment.loc[seg]
            sev_escalated = sev * cum_infl
            expected_claims_seg = policies * freq * sev_escalated * exposure_factor
            expected_premium_seg = policies * pp_per_policy.loc[seg] * exposure_factor
            expected_claims += expected_claims_seg
            expected_premium += expected_premium_seg
        loss_ratio = expected_claims / expected_premium if expected_premium > 0 else np.nan
        rows.append({"year": y, "expected_claims": expected_claims, "expected_premium": expected_premium, "loss_ratio": loss_ratio})
    return pd.DataFrame(rows)

def create_transformed_features(df):

    if 'cargo_value' in df.columns:
        df['log_cargo_value'] = np.log(df['cargo_value'] + 1)

    if 'weight' in df.columns:
        df['log_weight'] = np.log(df['weight'] + 1)

    if 'exposure' in df.columns:
        df['log_exposure'] = np.log(df['exposure'] + 1)

    return df

# ------------------------- Product processors -------------------------

def process_product(product_name: str, file_path: str,
                    freq_sheet: str, sev_sheet: str, segment_key: str, freq_formula_terms: list, sev_formula_terms: list, id_col: str = None) -> Dict:
    """General pipeline to:
    - read freq & sev
    - fit freq and sev models
    - produce per-segment pure premium, gross premium, limits
    - build projection for 5 years

    segment_key: column name used for grouping segments (e.g., 'segment' or 'group_number')
    freq_formula_terms, sev_formula_terms: lists of column names to use as model predictors (will be joined in formula)
    """
    print(f"Processing product: {product_name}")
    df_freq = read_excel_safe(file_path, freq_sheet)
    df_sev = read_excel_safe(file_path, sev_sheet)


    df_freq = create_transformed_features(df_freq)
    df_sev = create_transformed_features(df_sev)

    # Ensure numeric where expected
    ensure_numeric(df_freq, ['claim_count'])
    ensure_numeric(df_sev, ['claim_amount'])

    # compute counts per segment
    n_freq = df_freq.groupby(segment_key).size().rename('n_freq')
    n_sev = df_sev.groupby(segment_key).size().rename('n_sev')

    # merge counts into a segment-level frame
    segments = pd.DataFrame(pd.concat([n_freq, n_sev], axis=1)).fillna(0).astype(int)
    segments.index.name = segment_key

    # frequency modelling at observation-level: ensure claim_count exists
    if 'claim_count' not in df_freq.columns:
        # if not present, create a synthetic column: assume row == 1 policy and claim_count either 0/1 from claim flag
        df_freq['claim_count'] = df_freq.get('claim_count_same', 0)

    # build formulas (R-style) - response on left
    freq_predictors = ' + '.join(freq_formula_terms)
    freq_formula = f'claim_count ~ {freq_predictors}'
    freq_model_res, freq_model_type = fit_frequency_negative_binomial(df_freq, freq_formula)

    # severity
    sev_predictors = ' + '.join(sev_formula_terms)
    sev_formula = f'claim_amount ~ {sev_predictors}'
    sev_model_res, sev_model_type = fit_severity_gamma(df_sev, sev_formula)

    # For segment-level predictions we aggregate predictor variables by segment (means for numeric, modes for categorical)
    seg_agg = {}
    grouped_freq = df_freq.groupby(segment_key)
    freq_terms_clean = clean_formula_terms(freq_formula_terms)
    for col in freq_terms_clean:
        if df_freq[col].dtype.kind in 'biufc':
            seg_agg[col] = grouped_freq[col].mean()
        else:
            seg_agg[col] = grouped_freq[col].agg(lambda x: x.mode().iloc[0] if len(x.mode())>0 else x.iloc[0])
    seg_df = pd.DataFrame(seg_agg)
    seg_df.index.name = segment_key

    # Predict frequency per policy (exposure=1)
    try:
        if freq_model_type == 'nb':
            freq_pred = freq_model_res.predict(seg_df)
        else:
            freq_pred = freq_model_res.predict(seg_df)
            # if poisson with dispersion, keep raw prediction but user aware
    except Exception:
        # fallback to empirical frequency per segment (claim_count / rows)
        empirical = df_freq.groupby(segment_key)['claim_count'].sum() / df_freq.groupby(segment_key).size()
        freq_pred = empirical.reindex(seg_df.index).fillna(empirical.mean())

    # Predict severity per claim
    try:
        if sev_model_type == 'gamma':
            sev_pred = sev_model_res.predict(seg_df)
        elif sev_model_type == 'lognormal_ols':
            logpred = sev_model_res.predict(seg_df)
            sev_pred = np.exp(logpred)
        else:
            # empirical mean severity per segment
            sev_pred = df_sev.groupby(segment_key)['claim_amount'].mean().reindex(seg_df.index).fillna(df_sev['claim_amount'].mean())
    except Exception:
        sev_pred = df_sev.groupby(segment_key)['claim_amount'].mean().reindex(seg_df.index).fillna(df_sev['claim_amount'].mean())

    # pure premium per policy (exposure=1)
    pure_prem = freq_pred * sev_pred

    # classify risk
    risk_class = classify_risk(pure_prem)

    # gross premium
    gross_prem = pure_prem.apply(lambda x: compute_gross_premium(x, LOADINGS))

    # suggested limits
    limits_df = suggest_limits_from_severity(df_sev, sev_model_res, segment_key, None).set_index(segment_key)

    # compile segment summary
    seg_summary = pd.DataFrame({
        'n_freq': segments['n_freq'] if 'n_freq' in segments else 0,
        'n_sev': segments['n_sev'] if 'n_sev' in segments else 0,
        'freq_pred_per_policy': freq_pred,
        'sev_pred_per_claim': sev_pred,
        'pure_premium_per_policy': pure_prem,
        'gross_premium_per_policy': gross_prem,
        'risk_class': risk_class,
    }).join(limits_df)

    seg_summary = seg_summary.fillna(0)

    # Projection: prepare inputs
    inflation_proj = estimate_inflation_projection(PROJECTION_YEARS, INFLATION_HISTORY)
    exposure_growth = estimate_exposure_growth(COMPANY_FINANCIALS)

    # For projection we treat n_policies_by_segment = n_freq (assuming each freq row is one policy)
    n_policies_by_segment = seg_summary['n_freq'].replace(0, 1)  # avoid zero exposures for projection

    proj_df = project_loss_ratios(
        pp_per_policy=seg_summary['gross_premium_per_policy'],
        n_policies_by_segment=n_policies_by_segment,
        freq_pred_by_segment=seg_summary['freq_pred_per_policy'],
        sev_pred_by_segment=seg_summary['sev_pred_per_claim'],
        proj_years=PROJECTION_YEARS,
        inflation_proj=inflation_proj,
        exposure_growth=exposure_growth,
        loadings=LOADINGS
    )

    # Save excel output for product
    out_path = os.path.join(OUTPUT_DIR, f"pricing_{product_name}.xlsx")
    with pd.ExcelWriter(out_path, engine='xlsxwriter') as writer:
        # Write original frequency & severity raw tables
        df_freq.to_excel(writer, sheet_name='freq_raw', index=False)
        df_sev.to_excel(writer, sheet_name='sev_raw', index=False)
        # segment summary
        seg_summary.reset_index().to_excel(writer, sheet_name='segment_summary', index=False)
        proj_df.to_excel(writer, sheet_name='projections', index=False)

    result = {
        'product': product_name,
        'segment_summary': seg_summary,
        'projections': proj_df,
        'output_path': out_path,
        'freq_model_type': freq_model_type,
        'sev_model_type': sev_model_type,
    }
    print(f"--> saved output to {out_path}")
    return result


# ------------------------- Product-specific calls -------------------------

def main():
    # Map product specifics (columns provided in user brief)
    results = {}

    # 1) Cargo loss
    res_cargo = process_product(
    product_name='cargo_loss',
    file_path=FILES['cargo']['file'],
    freq_sheet=FILES['cargo']['freq_sheet'],
    sev_sheet=FILES['cargo']['sev_sheet'],
        segment_key='group_number',
       freq_formula_terms=[
    'log_cargo_value',
    'log_weight',
    'route_risk',
    'pilot_experience',
    'debris_density'
       ],
    sev_formula_terms=[
    'log_cargo_value',
    'log_weight',
    'route_risk',
    'pilot_experience',
    'debris_density'
    ]
    )
    results['cargo_loss'] = res_cargo

    # 2) Equipment failure
    res_equipment = process_product(
        product_name='equipment_failure',
        file_path = FILES['equipment']['file'],
        freq_sheet = FILES['equipment']['freq_sheet'],
        sev_sheet = FILES['equipment']['sev_sheet'],
        segment_key='segment',
        freq_formula_terms=['C(equipment_type)', 'C(solar_system)', 'exposure', 'C(usage_int_band)', 'C(equipment_age_band)'],
        sev_formula_terms=['C(equipment_type)', 'C(solar_system)', 'exposure', 'C(usage_int_band)', 'C(equipment_age_band)'],
    )
    results['equipment_failure'] = res_equipment

    # 3) Worker compensation
    res_worker = process_product(
        product_name='worker_compensation',
        file_path=FILES['worker']['file'],
        freq_sheet=FILES['worker']['freq_sheet'],
        sev_sheet=FILES['worker']['sev_sheet'],
        segment_key='segment',
        freq_formula_terms=['C(station_id)', 'C(occupation)', 'accident_history_flag', 'protective_gear_quality', 'exposure'],
        sev_formula_terms=['C(station_id)', 'C(occupation)', 'accident_history_flag', 'protective_gear_quality', 'exposure'],
    )
    results['worker_compensation'] = res_worker

    # 4) Business interruption (BI)
    res_bi = process_product(
        product_name='business_interruption',
        file_path=FILES['bi']['file'],
        freq_sheet = FILES['bi']['freq_sheet'],
        sev_sheet = FILES['bi']['sev_sheet'],
        segment_key='group_number',
        freq_formula_terms=['C(station_id)', 'C(solar_system)', 'supply_chain_index', 'avg_crew_exp', 'maintenance_freq', 'exposure'],
        sev_formula_terms=['C(station_id)', 'C(solar_system)', 'supply_chain_index', 'avg_crew_exp', 'maintenance_freq', 'exposure'],
    )
    results['business_interruption'] = res_bi

    # Summary report
    summary_rows = []
    for k, v in results.items():
        seg = v['segment_summary']
        total_policies = seg['n_freq'].sum()
        avg_loss_ratio_2175 = v['projections'].set_index('year').loc[PROJECTION_YEARS[0]]['loss_ratio']
        summary_rows.append({
            'product': k,
            'n_segments': seg.shape[0],
            'total_policies': int(total_policies),
            'proj_loss_ratio_2175': float(avg_loss_ratio_2175) if not pd.isna(avg_loss_ratio_2175) else None,
            'output_file': v['output_path']
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(os.path.join(OUTPUT_DIR, 'pricing_summary.xlsx'), index=False)
    print("All done. Summary saved to outputs/pricing_summary.xlsx")


if __name__ == '__main__':
    main()